# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nazama-tech/Flyrank-Ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task type: Classification — a yes/no guess.
For every piece of content, I want the model to answer one question: "Will this be a hit or not?" That's a yes/no answer, not a group, not a sorted list, not a number, which is what makes it "classification."

In [3]:
# This cell is for CODE (numbers, a query, a check).
task_type = "classification"
print(f"Task_type:",task_type)

Task_type: classification


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My model will predict whether a piece of content is a "hit" (`is_hit`). This label is defined by a rule: content is considered a 'hit' if its `pageviews_90d` are greater than or equal to the 90th percentile of all `pageviews_90d`. This is a **defined rule** based on an **observed outcome** (`pageviews_90d`) to classify content as a 'hit' or 'not a hit'.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Display the value counts of the target variable and its mean
cutoff = df["pageviews_90d"].quantile(0.90)
df["is_hit"] = (df["pageviews_90d"] >= cutoff).astype(int)

display(df['is_hit'].value_counts())
print(f"'Hit' rate: {df['is_hit'].mean():.3f}")

,count
is_hit,
0,26982
1,3018


'Hit' rate: 0.101


## 3. Success metric

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Success Metric: Precision**

For this classification problem, where we aim to predict whether content will be a 'hit' or not, **Precision** is a crucial success metric. Precision measures the proportion of content predicted as 'hits' that truly turn out to be 'hits'.

**Why Precision?**
*   **Minimizing Wasted Resources (False Positives):** A false positive occurs when the model predicts content will be a 'hit', but it actually isn't. If we use the model's predictions to decide which content to promote or invest in, a low precision means we would be allocating valuable resources (e.g., marketing budget, editorial attention) to content that won't deliver the desired 'hit' performance. High precision ensures that when we act on a 'hit' prediction, we are likely to be correct, thus optimizing resource allocation.
*   **Focus on Trustworthiness of Positive Predictions:** In a scenario where 'hits' are rare (our 'hit' rate is ~10%), it's especially important that the positive predictions made by the model are reliable. If the model says something is a hit, we want to trust that assessment.

While Precision is primary for efficient resource allocation, it's important to also consider **Recall** (the proportion of actual 'hits' that the model successfully identifies) to ensure we are not missing too many valuable opportunities. The **F1-score**, which is the harmonic mean of precision and recall, would provide a balanced view, but for a single 'defensible' metric, Precision addresses the direct business cost of acting on incorrect positive predictions.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# The baseline to beat: how often would a RANDOM guess be right?
base_rate = df["is_hit"].mean()
print(f"Real hit rate in the data: {base_rate*100:.1f}%")
print(f"-> A model guessing at random would only be right about {base_rate*100:.1f}% of the time it says 'yes'.")
print(f"-> That's the number my model's precision has to clearly beat to be worth using.")
success_metric = "precision"
print(f"Success Metric: {success_metric.capitalize()}")

Real hit rate in the data: 10.1%
-> A model guessing at random would only be right about 10.1% of the time it says 'yes'.
-> That's the number my model's precision has to clearly beat to be worth using.
Success Metric: Precision


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis for this task is a **piece of content**. Each row in our DataFrame `df` represents a unique piece of content, identified by its `content_id`.

This aligns perfectly with our classification task, where we aim to predict for *each individual piece of content* whether it will be a 'hit' or not. The features in each row describe that specific piece of content, and the `is_hit` target variable indicates its outcome.

In [10]:
# This cell is for CODE (numbers, a query, a check).

# Display the first few rows of the DataFrame to show the unit of analysis
print("First 5 rows of the DataFrame (each row is a piece of content):")
display(df.head())

First 5 rows of the DataFrame (each row is a piece of content):


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_hit
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,0
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,0
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,0
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


In [11]:
# Show the unit of analysis: one row = one content item
print("Shape:", df.shape)
print()
print("One row = one content item (one article/page), from one of", df['client_id'].nunique(), "clients.")
print()
df[["content_id", "client_id", "content_type", "main_intent", "word_count", "pageviews_90d", "is_hit"]].head(5)

Shape: (30000, 45)

One row = one content item (one article/page), from one of 32 clients.



,content_id,client_id,content_type,main_intent,word_count,pageviews_90d,is_hit
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3221.0,22,0
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,2481.0,10,0
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,3515.0,14,0
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,NaN,87,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,2803.0,177,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

I actually tested a real, plausible human rule ("high search volume + long content = hit") instead of just asserting ML is better

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Test a plausible human-written fixed rule: high search volume + long-form = 'hit'
sv_cutoff = df["search_volume"].median()
rule_pred = ((df["search_volume"] > sv_cutoff) & (df["word_count"] > 2000)).astype(int)

precision = (rule_pred & df["is_hit"]).sum() / rule_pred.sum()
recall = (rule_pred & df["is_hit"]).sum() / df["is_hit"].sum()

print("Rows the rule flags as 'hit':", rule_pred.sum())
print(f"Rule precision: {precision*100:.1f}%  (random baseline was {df['is_hit'].mean()*100:.1f}%)")
print(f"Rule recall: {recall*100:.1f}%  (share of real hits the rule actually caught)")

Rows the rule flags as 'hit': 4306
Rule precision: 9.7%  (random baseline was 10.1%)
Rule recall: 13.9%  (share of real hits the rule actually caught)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.